# Project Update 1 — Data Exploration & Preprocessing

This notebook prepares the **Yelp** and **XSum** subsets of the *Ship of Theseus Paraphrased Corpus* for downstream similarity and stylometric analysis.

### Goals
1. Inspect dataset structure and available paraphrase versions.
2. Focus the main analysis on texts with `source == "Human"`.
3. Build complete **T0 → T1 → T2 → T3** chains for:
   - ChatGPT
   - PaLM
   - Pegasus(full)
   - Dipper
4. Save clean processed datasets for teammates.
5. Produce dataset statistics for the report and presentation.

> Reusable preprocessing logic is kept in `src/preprocessing.py`. This notebook focuses on data inspection, validation, and output generation.


## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

# Allow imports from the project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import process_dataset

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)


Project root: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4
Raw data directory: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/raw
Processed data directory: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/processed


## 2. Load and Inspect Yelp

In [2]:
yelp_path = RAW_DIR / "paraphrased_datasets" / "yelp_paraphrased.csv"

print("Yelp path:", yelp_path)
print("File exists:", yelp_path.exists())

yelp = pd.read_csv(yelp_path)

print("Shape:", yelp.shape)
yelp.head()


Yelp path: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/raw/paraphrased_datasets/yelp_paraphrased.csv
File exists: True
Shape: (72847, 4)


,source,key,text,version_name
0,PaLM,yelp-494,sorry to say I did not have a good experience ...,original
1,BigScience,yelp-494,sorry to say I did not have a good experience ...,original
2,LLAMA,yelp-494,sorry to say I did not have a good experience ...,original
3,Eleuther-AI,yelp-494,sorry to say I did not have a good experience ...,original
4,OpenAI,yelp-494,sorry to say I did not have a good experience ...,original


### 2.1 Dataset schema

In [3]:
print("Columns:", yelp.columns.tolist())
print("\nData types:")
print(yelp.dtypes)

print("\nMissing values:")
print(yelp.isna().sum())


Columns: ['source', 'key', 'text', 'version_name']

Data types:
source          str
key             str
text            str
version_name    str
dtype: object

Missing values:
source          0
key             0
text            0
version_name    0
dtype: int64


### 2.2 Source categories

In [4]:
print("Source counts:")
print(yelp["source"].value_counts())

print("\nNumber of unique keys:", yelp["key"].nunique())


Source counts:
source
Human          10781
OpenAI         10719
LLAMA          10611
Tsinghua       10423
PaLM           10417
BigScience     10032
Eleuther-AI     9864
Name: count, dtype: int64

Number of unique keys: 491


### 2.3 Available paraphrase versions

In [5]:
print("Unique version names:")
for version in sorted(yelp["version_name"].dropna().unique()):
    print(version)


Unique version names:
chatgpt
chatgpt_chatgpt
chatgpt_chatgpt_chatgpt
dipper
dipper(high)
dipper(high)_dipper(high)
dipper(high)_dipper(high)_dipper(high)
dipper(low)
dipper(low)_dipper(low)
dipper(low)_dipper(low)_dipper(low)
dipper_dipper
dipper_dipper_dipper
original
palm
palm_palm
palm_palm_palm
pegasus(full)
pegasus(full)_pegasus(full)
pegasus(full)_pegasus(full)_pegasus(full)
pegasus(slight)
pegasus(slight)_pegasus(slight)
pegasus(slight)_pegasus(slight)_pegasus(slight)


### 2.4 Update 1 analysis scope

For the primary Update 1 analysis, we use four canonical paraphrasing chains:

| Paraphraser | T1 | T2 | T3 |
|---|---|---|---|
| ChatGPT | `chatgpt` | `chatgpt_chatgpt` | `chatgpt_chatgpt_chatgpt` |
| PaLM | `palm` | `palm_palm` | `palm_palm_palm` |
| Pegasus | `pegasus(full)` | `pegasus(full)_pegasus(full)` | `pegasus(full)_pegasus(full)_pegasus(full)` |
| Dipper | `dipper` | `dipper_dipper` | `dipper_dipper_dipper` |

`original` is used as **T0** for every chain.

The corpus also contains `pegasus(slight)`, `dipper(low)`, and `dipper(high)` variants. These are not included in the main Update 1 baseline and can be examined later as additional robustness conditions.

Because the project studies erosion of **human authorial identity**, the primary analysis uses rows where `source == "Human"`.


In [6]:
yelp_human = yelp[yelp["source"] == "Human"].copy()

print("Human-source rows:", len(yelp_human))
print("Human unique keys:", yelp_human["key"].nunique())


Human-source rows: 10781
Human unique keys: 491


## 3. Process Yelp into Complete T0–T3 Chains

In [7]:
yelp_processed = process_dataset(
    yelp,
    "yelp"
)

print("\nProcessed Yelp shape:", yelp_processed.shape)
print("\nComplete chains by paraphraser:")
print(yelp_processed["paraphraser"].value_counts())


yelp / ChatGPT | candidates: 491 | complete: 488
yelp / PaLM | candidates: 491 | complete: 486
yelp / Pegasus | candidates: 491 | complete: 491
yelp / Dipper | candidates: 491 | complete: 491

Processed Yelp shape: (1956, 8)

Complete chains by paraphraser:
paraphraser
Pegasus    491
Dipper     491
ChatGPT    488
PaLM       486
Name: count, dtype: int64


### 3.1 Validate processed Yelp data

In [8]:
print("Missing T0-T3 values:")
print(yelp_processed[["t0", "t1", "t2", "t3"]].isna().sum())

duplicate_count = yelp_processed.duplicated(
    subset=["dataset", "key", "paraphraser"]
).sum()

print("\nDuplicate dataset-key-paraphraser rows:", duplicate_count)


Missing T0-T3 values:
t0    0
t1    0
t2    0
t3    0
dtype: int64

Duplicate dataset-key-paraphraser rows: 0


### 3.2 Inspect one complete example

In [9]:
example = yelp_processed.iloc[0]

print("Dataset:", example["dataset"])
print("Key:", example["key"])
print("Paraphraser:", example["paraphraser"])

print("\nT0 ORIGINAL:\n", example["t0"])
print("\nT1:\n", example["t1"])
print("\nT2:\n", example["t2"])
print("\nT3:\n", example["t3"])


Dataset: yelp
Key: yelp-17670
Paraphraser: ChatGPT

T0 ORIGINAL:
 If I could give them negative starts I would. Worst customer service I have ever experienced.

T1:
 I would rate them with negative stars if possible. Their customer service is the worst I have ever encountered.

T2:
 If it were possible, I would give them a negative star rating. Their customer service is the most terrible I have ever experienced.

T3:
 If it were feasible, I would assign them a rating below zero stars. Their customer service is the worst I have ever encountered.


### 3.3 Yelp chain statistics

In [10]:
yelp_stats = (
    yelp_processed
    .groupby("paraphraser")
    .size()
    .reset_index(name="complete_chains")
    .sort_values("paraphraser")
)

yelp_stats


,paraphraser,complete_chains
0,ChatGPT,488
1,Dipper,491
2,PaLM,486
3,Pegasus,491


## 4. Load and Inspect XSum

In [11]:
xsum_path = RAW_DIR / "paraphrased_datasets" / "xsum_paraphrased.csv"

print("XSum path:", xsum_path)
print("File exists:", xsum_path.exists())

xsum = pd.read_csv(xsum_path)

print("Shape:", xsum.shape)
print("Columns:", xsum.columns.tolist())
print("\nSource counts:")
print(xsum["source"].value_counts())
print("\nUnique keys:", xsum["key"].nunique())


XSum path: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/raw/paraphrased_datasets/xsum_paraphrased.csv
File exists: True
Shape: (70388, 4)
Columns: ['source', 'key', 'text', 'version_name']

Source counts:
source
OpenAI         10413
LLAMA          10371
Tsinghua       10294
BigScience     10274
Human          10054
PaLM            9726
Eleuther-AI     9256
Name: count, dtype: int64

Unique keys: 476


In [12]:
xsum_human = xsum[xsum["source"] == "Human"].copy()

print("Human-source rows:", len(xsum_human))
print("Human unique keys:", xsum_human["key"].nunique())


Human-source rows: 10054
Human unique keys: 476


## 5. Process XSum into Complete T0–T3 Chains

In [13]:
xsum_processed = process_dataset(
    xsum,
    "xsum"
)

print("\nProcessed XSum shape:", xsum_processed.shape)
print("\nComplete chains by paraphraser:")
print(xsum_processed["paraphraser"].value_counts())


xsum / ChatGPT | candidates: 476 | complete: 471
xsum / PaLM | candidates: 476 | complete: 340
xsum / Pegasus | candidates: 476 | complete: 475
xsum / Dipper | candidates: 476 | complete: 476

Processed XSum shape: (1762, 8)

Complete chains by paraphraser:
paraphraser
Dipper     476
Pegasus    475
ChatGPT    471
PaLM       340
Name: count, dtype: int64


### Data availability note

The number of complete T0–T3 chains can differ across paraphrasers. In particular, XSum contains fewer complete PaLM chains than the other paraphrasers. This is treated as a **corpus completeness / sample availability issue**, not as evidence of model quality.

Only complete T0–T3 chains are retained so that each comparison uses documents with all four stages available.


### 5.1 Validate processed XSum data

In [14]:
print("Missing T0-T3 values:")
print(xsum_processed[["t0", "t1", "t2", "t3"]].isna().sum())

duplicate_count = xsum_processed.duplicated(
    subset=["dataset", "key", "paraphraser"]
).sum()

print("\nDuplicate dataset-key-paraphraser rows:", duplicate_count)


Missing T0-T3 values:
t0    0
t1    0
t2    0
t3    0
dtype: int64

Duplicate dataset-key-paraphraser rows: 0


## 6. Combine Yelp and XSum

In [15]:
update1_processed = pd.concat(
    [yelp_processed, xsum_processed],
    ignore_index=True
)

print("Combined shape:", update1_processed.shape)

print("\nComplete chains by dataset and paraphraser:")
print(
    update1_processed
    .groupby(["dataset", "paraphraser"])
    .size()
)


Combined shape: (3718, 8)

Complete chains by dataset and paraphraser:
dataset  paraphraser
xsum     ChatGPT        471
         Dipper         476
         PaLM           340
         Pegasus        475
yelp     ChatGPT        488
         Dipper         491
         PaLM           486
         Pegasus        491
dtype: int64


### 6.1 Final validation

In [16]:
print("Missing values in T0-T3:")
print(update1_processed[["t0", "t1", "t2", "t3"]].isna().sum())

print(
    "\nDuplicate dataset-key-paraphraser rows:",
    update1_processed.duplicated(
        subset=["dataset", "key", "paraphraser"]
    ).sum()
)

print("\nOutput columns:")
print(update1_processed.columns.tolist())


Missing values in T0-T3:
t0    0
t1    0
t2    0
t3    0
dtype: int64

Duplicate dataset-key-paraphraser rows: 0

Output columns:
['dataset', 'key', 'source', 'paraphraser', 't0', 't1', 't2', 't3']


## 7. Save Processed Datasets

In [17]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

yelp_processed.to_csv(
    PROCESSED_DIR / "yelp_update1.csv",
    index=False
)

xsum_processed.to_csv(
    PROCESSED_DIR / "xsum_update1.csv",
    index=False
)

update1_processed.to_csv(
    PROCESSED_DIR / "update1_processed.csv",
    index=False
)

print("Saved:")
print(PROCESSED_DIR / "yelp_update1.csv")
print(PROCESSED_DIR / "xsum_update1.csv")
print(PROCESSED_DIR / "update1_processed.csv")


Saved:
/Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/processed/yelp_update1.csv
/Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/processed/xsum_update1.csv
/Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/processed/update1_processed.csv


### Handoff file for downstream experiments

Members working on similarity metrics and stylometry should use:

```text
data/processed/update1_processed.csv
```

Schema:

```text
dataset | key | source | paraphraser | t0 | t1 | t2 | t3
```

This ensures all downstream analyses use the same preprocessing and sample population.


## 8. Create Dataset Statistics

In [18]:
dataset_stats = (
    update1_processed
    .groupby(["dataset", "paraphraser"])
    .size()
    .reset_index(name="complete_chains")
    .sort_values(["dataset", "paraphraser"])
)

dataset_stats


,dataset,paraphraser,complete_chains
0,xsum,ChatGPT,471
1,xsum,Dipper,476
2,xsum,PaLM,340
3,xsum,Pegasus,475
4,yelp,ChatGPT,488
5,yelp,Dipper,491
6,yelp,PaLM,486
7,yelp,Pegasus,491


In [19]:
original_stats = pd.DataFrame({
    "dataset": ["yelp", "xsum"],
    "human_unique_keys": [
        yelp_human["key"].nunique(),
        xsum_human["key"].nunique(),
    ]
})

original_stats


,dataset,human_unique_keys
0,yelp,491
1,xsum,476


### 8.1 Save metadata tables

In [20]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)

dataset_stats.to_csv(
    METADATA_DIR / "update1_dataset_statistics.csv",
    index=False
)

original_stats.to_csv(
    METADATA_DIR / "update1_original_document_counts.csv",
    index=False
)

print("Saved metadata:")
print(METADATA_DIR / "update1_dataset_statistics.csv")
print(METADATA_DIR / "update1_original_document_counts.csv")


Saved metadata:
/Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/metadata/update1_dataset_statistics.csv
/Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/metadata/update1_original_document_counts.csv


## 9. Validation

In [21]:
print("Final shape:", update1_processed.shape)

print("\nColumns:")
print(update1_processed.columns.tolist())

print("\nMissing T0-T3 values:")
print(
    update1_processed[
        ["t0", "t1", "t2", "t3"]
    ].isna().sum()
)

duplicate_count = update1_processed.duplicated(
    subset=["dataset", "key", "paraphraser"]
).sum()

print("\nDuplicate rows:", duplicate_count)

# Verify the saved file exists
output_path = PROCESSED_DIR / "update1_processed.csv"

print("\nOutput file exists:", output_path.exists())
print("Output path:", output_path)

Final shape: (3718, 8)

Columns:
['dataset', 'key', 'source', 'paraphraser', 't0', 't1', 't2', 't3']

Missing T0-T3 values:
t0    0
t1    0
t2    0
t3    0
dtype: int64

Duplicate rows: 0

Output file exists: True
Output path: /Users/chuweicai/Desktop/6120/Ship-of-Theseus-Paraphrased-Corpus-CS6120-Team4/data/processed/update1_processed.csv


## 10. Preprocessing Summary

The Update 1 preprocessing pipeline:

1. Loads Yelp and XSum paraphrased datasets.
2. Focuses on `source == "Human"` for the primary authorial-identity analysis.
3. Uses four canonical paraphrasers: ChatGPT, PaLM, Pegasus(full), and Dipper.
4. Organizes each document into `T0`, `T1`, `T2`, and `T3`.
5. Removes incomplete chains.
6. Validates missing values and duplicate document/paraphraser pairs.
7. Saves one standardized file for all downstream experiments.

The next project steps can therefore operate directly on `data/processed/update1_processed.csv`.
